In [ ]:
import os
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, utils, models
from PIL import Image
import copy
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_recall_curve, roc_auc_score
from sklearn.metrics import classification_report
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, KFold
from collections import defaultdict, Counter
from facenet_pytorch import InceptionResnetV1
from models_vggface.vgg_face import vgg_face, VggFaceFeatureExtractor
import clip


In [2]:
class RFDDataset(Dataset):
    def __init__(self, img_paths, disease_labels, transform=None):
        self.img_paths = img_paths
        self.disease_labels = disease_labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        label = self.disease_labels[idx]
        image = Image.open(img_path).convert("RGB")  # Convert to RGB

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# Load metadata
img_dir = "data/rd_images"
data = pd.read_csv("data/disease_images.csv")

img_names = data["image_name"]
disease_names = data["disease_name"]
disease_abbr = data["disease_abbr"]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img_paths = [os.path.join(img_dir, disease, name) for disease, name in zip(disease_abbr, img_names)]
disease_mappings = {disease: i for i, disease in enumerate(disease_names.unique())}
disease_labels = [disease_mappings[disease] for disease in disease_names]

'''
Below is to make sure that classes with single sample will be placed into training set for sure
'''

# Step 1: Group image indices by class
class_to_indices = defaultdict(list)
for idx, label in enumerate(disease_labels):
    class_to_indices[label].append(idx)

singleton_indices = [idxs[0] for idxs in class_to_indices.values() if len(idxs) == 1]
non_singleton_indices = [idx for idxs in class_to_indices.values() if len(idxs) > 1 for idx in idxs]

# Create the singleton part of training set
singleton_imgs = [img_paths[i] for i in singleton_indices]
singleton_labels = [disease_labels[i] for i in singleton_indices]

# For non-singleton samples, split into train/test (stratified)
non_singleton_imgs = [img_paths[i] for i in non_singleton_indices]
non_singleton_labels = [disease_labels[i] for i in non_singleton_indices]

train_ns_imgs, test_imgs, train_ns_labels, test_labels = train_test_split(
    non_singleton_imgs,
    non_singleton_labels,
    test_size=0.25,
    random_state=42,
    stratify=non_singleton_labels
)

# Final training set = singleton + non-singleton training
train_imgs = singleton_imgs + train_ns_imgs
train_labels = singleton_labels + train_ns_labels

# Step 2: Prepare data for k-fold on training set
train_indices = list(range(len(train_imgs)))
label_array = np.array(train_labels)

# Group by class
class_to_indices = defaultdict(list)
for idx, label in enumerate(label_array):
    class_to_indices[label].append(idx)

# Singleton detection
singleton_indices = [idxs[0] for idxs in class_to_indices.values() if len(idxs) == 1]
non_singleton_indices = [idx for idxs in class_to_indices.values() if len(idxs) > 1 for idx in idxs]
non_singleton_labels = [train_labels[idx] for idx in non_singleton_indices]

# Step 3: StratifiedKFold on non-singleton training data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = []
for train_idx_ns, val_idx_ns in skf.split(non_singleton_indices, non_singleton_labels):
    train_idx = [non_singleton_indices[i] for i in train_idx_ns] + singleton_indices
    val_idx = [non_singleton_indices[i] for i in val_idx_ns]
    folds.append((train_idx, val_idx))

# data sets
train_dataset = RFDDataset(train_imgs, train_labels, transform=transform)
test_dataset = RFDDataset(test_imgs, test_labels, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, drop_last=False)


/data/lab_ph/kathy/miniconda3/envs/RFD/lib/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


In [4]:
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Number of training samples: 343
Number of testing samples: 113


In [5]:
import torch
import torch.nn as nn
import torchvision.models as models

class RareDiseaseModel(nn.Module):
    def __init__(self, backbone, num_classes, use_se=True, dropout_rate=0.4, use_layernorm=False, conv_out_channels=None):
        """
        Args:
            backbone (nn.Module): Any feature extractor (EfficientNet, ResNet, ViT, etc.).
            num_classes (int): Number of output classes.
            use_se (bool): Whether to use Squeeze-and-Excitation (SE) block.
            dropout_rate (float): Dropout probability.
            use_layernorm (bool): Use LayerNorm instead of BatchNorm to avoid batch size issues.
        """
        super(RareDiseaseModel, self).__init__()

        self.backbone = backbone  # Store backbone
        self.use_se = use_se
        self.se_block_2d = None
        self.se_block_4d = None
        self.in_features = self._get_backbone_out_features()

        # Optional SE
        if use_se:
            self.se_block_2d = nn.Sequential(
                nn.Linear(self.in_features, self.in_features // 16),
                nn.ReLU(),
                nn.Linear(self.in_features // 16, self.in_features),
                nn.Sigmoid()
            )

            # Only define 4D SE block if conv_out_channels is provided
            if conv_out_channels is not None:
                self.se_block_4d = nn.Sequential(
                    nn.AdaptiveAvgPool2d(1),
                    nn.Conv2d(conv_out_channels, conv_out_channels // 16, kernel_size=1),
                    nn.ReLU(),
                    nn.Conv2d(conv_out_channels // 16, conv_out_channels, kernel_size=1),
                    nn.Sigmoid()
                )
            else:
                self.se_block_4d = None


        # Optional normalization
        norm_layer = nn.LayerNorm(self.in_features) if use_layernorm else nn.BatchNorm1d(self.in_features)

        # Classification head
        self.classifier = nn.Sequential(
            norm_layer,
            nn.Dropout(dropout_rate),
            nn.Linear(self.in_features, num_classes)
        )

    def _get_backbone_out_features(self):
        param = next(self.backbone.parameters())
        dummy_input = torch.randn(1, 3, 224, 224).to(device=param.device, dtype=param.dtype)

        with torch.no_grad():
            # ResNet-style
            if hasattr(self.backbone, 'fc') and isinstance(self.backbone.fc, nn.Linear):
                in_features = self.backbone.fc.in_features
                self.backbone.fc = nn.Identity()
                return in_features

            # CLIP ViT model (clip_model.visual)
            if type(self.backbone).__name__ == "VisionTransformer" and "clip" in str(type(self.backbone)).lower():
                features = self.backbone(dummy_input)
                return features.shape[1]

            # VGGFace (your custom conv feature extractor)
            if isinstance(self.backbone, VggFaceFeatureExtractor):
                out = self.backbone(dummy_input)
                out = torch.flatten(out, 1)
                return out.shape[1]

            # EfficientNet, DenseNet, Inception, etc.
            if hasattr(self.backbone, 'classifier'):
                classifier = self.backbone.classifier
                if isinstance(classifier, nn.Sequential):
                    for layer in reversed(classifier):
                        if isinstance(layer, nn.Linear):
                            in_features = layer.in_features
                            self.backbone.classifier = nn.Identity()
                            return in_features
                elif isinstance(classifier, nn.Linear):
                    in_features = classifier.in_features
                    self.backbone.classifier = nn.Identity()
                    return in_features
                else:
                    raise TypeError("Unknown classifier structure in `.classifier`")

            # Swin
            if hasattr(self.backbone, 'head') and isinstance(self.backbone.head, nn.Linear):
                in_features = self.backbone.head.in_features
                self.backbone.head = nn.Identity()
                return in_features

            # ViT from timm
            if hasattr(self.backbone, 'heads') and hasattr(self.backbone.heads, 'head'):
                in_features = self.backbone.heads.head.in_features
                self.backbone.heads.head = nn.Identity()
                return in_features

            # Fallback: forward + flatten
            features = self.backbone(dummy_input)
            if isinstance(features, tuple):
                features = features[0]
            if features.ndim == 4:
                features = torch.flatten(features, 1)
            return features.shape[1]

        # If nothing matches
        raise ValueError(f"Unsupported backbone architecture: {type(self.backbone)}")


    def forward(self, x):
        if hasattr(self.backbone, "encode_image"):
            features = self.backbone.encode_image(x)
        else:
            features = self.backbone(x)

        if isinstance(features, tuple):
            features = features[0]
        elif hasattr(features, 'logits'):
            features = features.logits

        # --- Apply SE block if needed ---
        if features.ndim == 4 and self.se_block_4d:
            se_weight = self.se_block_4d(features)
            features = features * se_weight
        elif features.ndim == 2 and self.se_block_2d:
            se_weight = self.se_block_2d(features)
            features = features * se_weight

        # --- Flatten if needed ---
        if features.ndim == 4:
            features = torch.flatten(features, 1)

        return self.classifier(features)




In [6]:
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []

    with tqdm(train_loader, unit="batch") as bar:
        for images, labels in bar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            bar.set_postfix(loss=loss.item())

    accuracy = accuracy_score(all_labels, all_preds) * 100

    return total_loss / len(train_loader), accuracy

def topk_accuracy(output, target, k=5):
    with torch.no_grad():
        topk_preds = torch.topk(output, k, dim=1).indices  # Get top-K class indices
        correct = topk_preds.eq(target.view(-1, 1).expand_as(topk_preds))  # Check if true label is in top-K
        return correct.any(dim=1).float().mean().item()  # Compute mean accuracy

def evaluate(model, test_loader, criterion, device, topk=[1, 5, 10, 30]):
    model.eval()
    total_loss = 0
    all_labels = []
    all_preds = []
    all_probs = []
    topk_accuracies = {k: 0 for k in topk}

    with torch.no_grad():
        for images, labels in tqdm(test_loader, unit="batch"):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = outputs.max(1)

            probabilities = F.softmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probabilities.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds, average="macro")

    # Top-K Accuracies (global, consistent with sklearn)
    all_outputs = torch.tensor(all_probs)
    all_labels_tensor = torch.tensor(all_labels)
    topk_accuracies = {k: topk_accuracy(all_outputs, all_labels_tensor, k=k) * 100 for k in topk}

    return total_loss / len(test_loader), accuracy, f1, topk_accuracies, all_labels, all_preds, all_probs


In [7]:
def train_model_on_folds(base_model, dataset, folds, save_dir, device, num_epochs=50):
    os.makedirs(save_dir, exist_ok=True)
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(folds):
        print(f"\n Fold {fold + 1}")
        best_acc = 0

        train_loader = DataLoader(Subset(dataset, train_idx), batch_size=32, shuffle=True, num_workers=4, drop_last=False)
        val_loader = DataLoader(Subset(dataset, val_idx), batch_size=32, shuffle=False, num_workers=4, drop_last=False)

        # Initialize model + training config
        model = copy.deepcopy(base_model)
        model.to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

        for epoch in range(num_epochs):
            print(f"\n Fold {fold + 1}, Epoch {epoch + 1}/{num_epochs}")
            train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(model, val_loader, criterion, device)

            scheduler.step()

            # check acc
            if val_acc >= best_acc:
                best_acc = val_acc
                
                print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}% | "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}%, F1: {val_f1:.4f}")

                print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

                torch.save(model.state_dict(), os.path.join(save_dir, f"fold{fold+1}_best.pth"))
                print(f"Best model saved with acc {best_acc:.4f}%")

        fold_metrics.append({
            "val_acc": best_acc,
            "val_topk": topk_accuracies
        })

    # Summary
    avg_acc = np.mean([m["val_acc"] for m in fold_metrics])
    avg_topk = {k: np.mean([m["val_topk"][k] for m in fold_metrics]) for k in fold_metrics[0]["val_topk"].keys()}

    print(f"\n Average across {len(folds)} folds:")
    print(f" - Accuracy: {avg_acc:.2f}%")
    print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in avg_topk.items()]))
    return fold_metrics


In [8]:
class RareDiseaseFaceNet(nn.Module):
    def __init__(self, num_classes, freeze_backbone=False, use_se=True, dropout_rate=0.5):
        super(RareDiseaseFaceNet, self).__init__()

        # Load FaceNet (pretrained on VGGFace2)
        self.backbone = InceptionResnetV1(pretrained='vggface2', classify=False)

        # Remove FaceNet’s last batch normalization layer
        self.backbone.last_bn = nn.Identity()  # This prevents the batch norm issue

        # Extract feature size (FaceNet outputs a 512D embedding)
        in_features = self.backbone.last_linear.in_features
        self.backbone.last_linear = nn.Identity()  # Remove default classification layer

        # Optional: Freeze the FaceNet backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Optional Squeeze-and-Excitation (SE) block for feature refinement
        if use_se:
            self.se_block = nn.Sequential(
                nn.Linear(in_features, in_features // 16),
                nn.ReLU(),
                nn.Linear(in_features // 16, in_features),
                nn.Sigmoid()
            )
        else:
            self.se_block = None

        # Custom classifier for rare disease classification
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)  # Extract FaceNet features
        if self.se_block:
            se_weight = self.se_block(features)
            features = features * se_weight  # Apply SE weighting

        output = self.classifier(features)  # Custom classification head
        return output


In [ ]:
# facenet
num_classes = len(disease_mappings)
save_dir = "logs_new/FACENET"
device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
model = RareDiseaseFaceNet(num_classes=num_classes, freeze_backbone=False, use_se=True, dropout_rate=0.5)
facenet_matrix = train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)


 Fold 1

 Fold 1, Epoch 1/50


100%|██████████| 3/3 [00:00<00:00,  4.96batch/s]


Epoch 1: Train Loss: 4.7463, Train Acc: 0.7246% | Val Loss: 4.6274, Val Acc: 2.9851%, F1: 0.0029
Top-1 Accuracy: 2.9851% | Top-5 Accuracy: 4.4776% | Top-10 Accuracy: 8.9552% | Top-30 Accuracy: 31.3433%
Best model saved with acc 2.9851%

 Fold 1, Epoch 2/50


100%|██████████| 3/3 [00:00<00:00,  5.95batch/s]


Epoch 2: Train Loss: 4.6623, Train Acc: 1.4493% | Val Loss: 4.6256, Val Acc: 2.9851%, F1: 0.0025
Top-1 Accuracy: 2.9851% | Top-5 Accuracy: 4.4776% | Top-10 Accuracy: 11.9403% | Top-30 Accuracy: 34.3284%
Best model saved with acc 2.9851%

 Fold 1, Epoch 3/50


...


 Fold 5, Epoch 49/50


100%|██████████| 3/3 [00:00<00:00,  5.48batch/s]



 Fold 5, Epoch 50/50


100%|██████████| 3/3 [00:00<00:00,  5.67batch/s]


 Average across 5 folds:
 - Accuracy: 9.67%
Top-1 Accuracy: 5.4365% | Top-5 Accuracy: 21.4473% | Top-10 Accuracy: 34.7535% | Top-30 Accuracy: 58.9055%


In [ ]:
# facenet test
test_metrics = []
for fold in range(5):
    num_classes = len(disease_mappings)
    device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
    test_model = RareDiseaseFaceNet(num_classes=num_classes).to(device)
    test_model.load_state_dict(torch.load(f'logs_new/FACENET/fold{fold+1}_best.pth'))
    test_model.eval()  # Set model to evaluation mode
    criterion = nn.CrossEntropyLoss()
    val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
    print(val_acc)
    print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

    test_metrics.append({
        "val_acc": val_acc,
        "val_topk": topk_accuracies
    })

# Compute average and std of overall accuracy
val_accs = [m["val_acc"] for m in test_metrics]
avg_acc = np.mean(val_accs)
std_acc = np.std(val_accs, ddof=1)

# Compute average and std for each top-k accuracy
avg_topk = {}
std_topk = {}
for k in test_metrics[0]["val_topk"].keys():
    scores = [m["val_topk"][k] for m in test_metrics]
    avg_topk[k] = np.mean(scores)
    std_topk[k] = np.std(scores, ddof=1)

# Print results
print(f"\nAverage across {len(test_metrics)} folds:")
print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

for k in sorted(avg_topk.keys(), key=int):
    print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")


100%|██████████| 4/4 [00:00<00:00,  4.90batch/s]


10.619469026548673
Top-1 Accuracy: 10.6195% | Top-5 Accuracy: 23.8938% | Top-10 Accuracy: 39.8230% | Top-30 Accuracy: 57.5221%


100%|██████████| 4/4 [00:00<00:00,  6.86batch/s]


7.964601769911504
Top-1 Accuracy: 7.9646% | Top-5 Accuracy: 18.5841% | Top-10 Accuracy: 32.7434% | Top-30 Accuracy: 54.8673%


100%|██████████| 4/4 [00:00<00:00,  7.26batch/s]


11.504424778761061
Top-1 Accuracy: 11.5044% | Top-5 Accuracy: 32.7434% | Top-10 Accuracy: 39.8230% | Top-30 Accuracy: 64.6018%


100%|██████████| 4/4 [00:00<00:00,  6.81batch/s]


11.504424778761061
Top-1 Accuracy: 11.5044% | Top-5 Accuracy: 26.5487% | Top-10 Accuracy: 32.7434% | Top-30 Accuracy: 61.9469%


100%|██████████| 4/4 [00:00<00:00,  7.26batch/s]

7.964601769911504
Top-1 Accuracy: 7.9646% | Top-5 Accuracy: 21.2389% | Top-10 Accuracy: 29.2035% | Top-30 Accuracy: 52.2124%

Average across 5 folds:
 - Accuracy: 9.91% ± 1.81% (1-sigma)
 - Top-1 Accuracy: 9.91% ± 1.81% (1-sigma)
 - Top-5 Accuracy: 24.60% ± 5.43% (1-sigma)
 - Top-10 Accuracy: 34.87% ± 4.75% (1-sigma)
 - Top-30 Accuracy: 58.23% ± 5.06% (1-sigma)


In [ ]:
# resnet152
num_classes = len(disease_mappings)
save_dir = "logs_new/RESNET152"
device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
backbone = models.resnet152(weights='ResNet152_Weights.IMAGENET1K_V2')
model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)


 Fold 1

 Fold 1, Epoch 1/50


100%|██████████| 3/3 [00:00<00:00,  4.63batch/s]


Epoch 1: Train Loss: 4.9456, Train Acc: 0.0000% | Val Loss: 4.6100, Val Acc: 1.4925%, F1: 0.0066
Top-1 Accuracy: 1.4925% | Top-5 Accuracy: 10.4478% | Top-10 Accuracy: 10.4478% | Top-30 Accuracy: 31.3433%
Best model saved with acc 1.4925%

 Fold 1, Epoch 2/50


100%|██████████| 3/3 [00:01<00:00,  2.40batch/s]



 Fold 1, Epoch 3/50


...

100%|██████████| 3/3 [00:00<00:00,  4.99batch/s]


Epoch 49: Train Loss: 0.0216, Train Acc: 100.0000% | Val Loss: 4.3672, Val Acc: 7.5758%, F1: 0.0521
Top-1 Accuracy: 7.5758% | Top-5 Accuracy: 22.7273% | Top-10 Accuracy: 28.7879% | Top-30 Accuracy: 54.5455%
Best model saved with acc 7.5758%

 Fold 5, Epoch 50/50


100%|██████████| 3/3 [00:00<00:00,  4.74batch/s]


Epoch 50: Train Loss: 0.0163, Train Acc: 100.0000% | Val Loss: 4.3741, Val Acc: 7.5758%, F1: 0.0528
Top-1 Accuracy: 7.5758% | Top-5 Accuracy: 24.2424% | Top-10 Accuracy: 27.2727% | Top-30 Accuracy: 54.5455%
Best model saved with acc 7.5758%

 Average across 5 folds:
 - Accuracy: 12.09%
Top-1 Accuracy: 10.8774% | Top-5 Accuracy: 25.0927% | Top-10 Accuracy: 33.5550% | Top-30 Accuracy: 54.9842%


[{'val_acc': 10.44776119402985,
  'val_topk': {1: 10.447761416435242,
   5: 19.402985274791718,
   10: 26.865673065185547,
   30: 55.22388219833374}},
 {'val_acc': 10.606060606060606,
  'val_topk': {1: 10.606060922145844,
   5: 21.212121844291687,
   10: 30.30303120613098,
   30: 46.9696968793869}},
 {'val_acc': 16.666666666666664,
  'val_topk': {1: 15.15151560306549,
   5: 34.84848439693451,
   10: 40.909090638160706,
   30: 56.060606241226196}},
 {'val_acc': 15.151515151515152,
  'val_topk': {1: 10.606060922145844,
   5: 25.757575035095215,
   10: 42.424243688583374,
   30: 62.12121248245239}},
 {'val_acc': 7.575757575757576,
  'val_topk': {1: 7.575757801532745,
   5: 24.242424964904785,
   10: 27.272728085517883,
   30: 54.54545617103577}}]

In [ ]:
# resnet152 test
torch.cuda.empty_cache()
test_metrics = []
for fold in range(5):
    num_classes = len(disease_mappings)
    device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
    backbone = models.resnet152(weights='ResNet152_Weights.IMAGENET1K_V2')
    test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
    test_model.load_state_dict(torch.load(f'logs_new/RESNET152/fold{fold+1}_best.pth'))
    test_model.to(device)
    test_model.eval()  # Set model to evaluation mode
    criterion = nn.CrossEntropyLoss()
    val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
    print(val_acc)
    print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

    test_metrics.append({
        "val_acc": val_acc,
        "val_topk": topk_accuracies
    })

# Compute average and std of overall accuracy
val_accs = [m["val_acc"] for m in test_metrics]
avg_acc = np.mean(val_accs)
std_acc = np.std(val_accs, ddof=1)

# Compute average and std for each top-k accuracy
avg_topk = {}
std_topk = {}
for k in test_metrics[0]["val_topk"].keys():
    scores = [m["val_topk"][k] for m in test_metrics]
    avg_topk[k] = np.mean(scores)
    std_topk[k] = np.std(scores, ddof=1)

# Print results
print(f"\nAverage across {len(test_metrics)} folds:")
print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

for k in sorted(avg_topk.keys(), key=int):
    print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")

100%|██████████| 4/4 [00:00<00:00,  5.40batch/s]


6.1946902654867255
Top-1 Accuracy: 6.1947% | Top-5 Accuracy: 20.3540% | Top-10 Accuracy: 31.8584% | Top-30 Accuracy: 51.3274%


100%|██████████| 4/4 [00:00<00:00,  5.81batch/s]


8.849557522123893
Top-1 Accuracy: 8.8496% | Top-5 Accuracy: 20.3540% | Top-10 Accuracy: 23.0089% | Top-30 Accuracy: 56.6372%


100%|██████████| 4/4 [00:00<00:00,  5.11batch/s]


7.964601769911504
Top-1 Accuracy: 7.9646% | Top-5 Accuracy: 21.2389% | Top-10 Accuracy: 31.8584% | Top-30 Accuracy: 52.2124%


100%|██████████| 4/4 [00:00<00:00,  6.47batch/s]


6.1946902654867255
Top-1 Accuracy: 6.1947% | Top-5 Accuracy: 16.8142% | Top-10 Accuracy: 27.4336% | Top-30 Accuracy: 55.7522%


100%|██████████| 4/4 [00:00<00:00,  6.04batch/s]

5.3097345132743365
Top-1 Accuracy: 5.3097% | Top-5 Accuracy: 14.1593% | Top-10 Accuracy: 28.3186% | Top-30 Accuracy: 55.7522%

Average across 5 folds:
 - Accuracy: 6.90% ± 1.45% (1-sigma)
 - Top-1 Accuracy: 6.90% ± 1.45% (1-sigma)
 - Top-5 Accuracy: 18.58% ± 3.00% (1-sigma)
 - Top-10 Accuracy: 28.50% ± 3.67% (1-sigma)
 - Top-30 Accuracy: 54.34% ± 2.39% (1-sigma)


In [ ]:
# densenet169
num_classes = len(disease_mappings)
save_dir = "logs_new/DENSENET169"
device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
backbone = models.densenet169(weights='DenseNet169_Weights.DEFAULT')
model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)



 Fold 1

 Fold 1, Epoch 1/50


100%|██████████| 3/3 [00:00<00:00,  3.82batch/s]


Epoch 1: Train Loss: 4.7514, Train Acc: 0.7246% | Val Loss: 4.6659, Val Acc: 0.0000%, F1: 0.0000
Top-1 Accuracy: 0.0000% | Top-5 Accuracy: 10.4478% | Top-10 Accuracy: 13.4328% | Top-30 Accuracy: 34.3284%
Best model saved with acc 0.0000%

 Fold 1, Epoch 2/50


100%|██████████| 3/3 [00:00<00:00,  4.40batch/s]


Epoch 2: Train Loss: 3.6636, Train Acc: 34.0580% | Val Loss: 4.6092, Val Acc: 4.4776%, F1: 0.0142
Top-1 Accuracy: 4.4776% | Top-5 Accuracy: 14.9254% | Top-10 Accuracy: 19.4030% | Top-30 Accuracy: 43.2836%
Best model saved with acc 4.4776%

 Fold 1, Epoch 3/50


...

100%|██████████| 3/3 [00:05<00:00,  1.98s/batch]



 Fold 5, Epoch 50/50


100%|██████████| 3/3 [00:00<00:00,  4.75batch/s]


 Average across 5 folds:
 - Accuracy: 17.22%
Top-1 Accuracy: 14.7987% | Top-5 Accuracy: 34.1610% | Top-10 Accuracy: 42.9127% | Top-30 Accuracy: 64.3464%


[{'val_acc': 16.417910447761194,
  'val_topk': {1: 16.417910158634186,
   5: 26.865673065185547,
   10: 38.805970549583435,
   30: 65.67164063453674}},
 {'val_acc': 24.242424242424242,
  'val_topk': {1: 22.727273404598236,
   5: 43.939393758773804,
   10: 53.03030014038086,
   30: 72.72727489471436}},
 {'val_acc': 22.727272727272727,
  'val_topk': {1: 18.18181872367859,
   5: 36.36363744735718,
   10: 45.45454680919647,
   30: 62.12121248245239}},
 {'val_acc': 12.121212121212121,
  'val_topk': {1: 7.575757801532745,
   5: 36.36363744735718,
   10: 43.939393758773804,
   30: 59.090906381607056}},
 {'val_acc': 10.606060606060606,
  'val_topk': {1: 9.090909361839294,
   5: 27.272728085517883,
   10: 33.33333432674408,
   30: 62.12121248245239}}]

In [ ]:
# densenet169 test
test_metrics = []
for fold in range(5):
    num_classes = len(disease_mappings)
    device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
    backbone = models.densenet169(weights='DenseNet169_Weights.DEFAULT')
    test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
    test_model.load_state_dict(torch.load(f'logs_new/DENSENET169/fold{fold+1}_best.pth'))
    test_model.to(device)
    test_model.eval()  # Set model to evaluation mode
    criterion = nn.CrossEntropyLoss()
    val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
    print(val_acc)
    print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

    test_metrics.append({
        "val_acc": val_acc,
        "val_topk": topk_accuracies
    })

# Compute average and std of overall accuracy
val_accs = [m["val_acc"] for m in test_metrics]
avg_acc = np.mean(val_accs)
std_acc = np.std(val_accs, ddof=1)

# Compute average and std for each top-k accuracy
avg_topk = {}
std_topk = {}
for k in test_metrics[0]["val_topk"].keys():
    scores = [m["val_topk"][k] for m in test_metrics]
    avg_topk[k] = np.mean(scores)
    std_topk[k] = np.std(scores, ddof=1)

# Print results
print(f"\nAverage across {len(test_metrics)} folds:")
print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

for k in sorted(avg_topk.keys(), key=int):
    print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")

100%|██████████| 4/4 [00:00<00:00,  4.24batch/s]


16.8141592920354
Top-1 Accuracy: 16.8142% | Top-5 Accuracy: 38.0531% | Top-10 Accuracy: 46.0177% | Top-30 Accuracy: 64.6018%


100%|██████████| 4/4 [00:00<00:00,  5.99batch/s]


15.04424778761062
Top-1 Accuracy: 15.0442% | Top-5 Accuracy: 30.9735% | Top-10 Accuracy: 40.7080% | Top-30 Accuracy: 64.6018%


100%|██████████| 4/4 [00:00<00:00,  6.33batch/s]


16.8141592920354
Top-1 Accuracy: 16.8142% | Top-5 Accuracy: 30.0885% | Top-10 Accuracy: 44.2478% | Top-30 Accuracy: 63.7168%


100%|██████████| 4/4 [00:00<00:00,  6.51batch/s]


18.58407079646018
Top-1 Accuracy: 18.5841% | Top-5 Accuracy: 37.1681% | Top-10 Accuracy: 44.2478% | Top-30 Accuracy: 61.9469%


100%|██████████| 4/4 [00:00<00:00,  6.58batch/s]

12.389380530973451
Top-1 Accuracy: 12.3894% | Top-5 Accuracy: 31.8584% | Top-10 Accuracy: 39.8230% | Top-30 Accuracy: 67.2566%

Average across 5 folds:
 - Accuracy: 15.93% ± 2.34% (1-sigma)
 - Top-1 Accuracy: 15.93% ± 2.34% (1-sigma)
 - Top-5 Accuracy: 33.63% ± 3.70% (1-sigma)
 - Top-10 Accuracy: 43.01% ± 2.63% (1-sigma)
 - Top-30 Accuracy: 64.42% ± 1.92% (1-sigma)


In [ ]:
# SWIN_T
num_classes = len(disease_mappings)
save_dir = "logs_new/SWIN_T"
device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
backbone = models.swin_t(weights="Swin_T_Weights.IMAGENET1K_V1")
model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)



 Fold 1

 Fold 1, Epoch 1/50


100%|██████████| 3/3 [00:00<00:00,  4.61batch/s]


Epoch 1: Train Loss: 4.9556, Train Acc: 1.4493% | Val Loss: 4.6718, Val Acc: 1.4925%, F1: 0.0059
Top-1 Accuracy: 1.4925% | Top-5 Accuracy: 5.9701% | Top-10 Accuracy: 10.4478% | Top-30 Accuracy: 26.8657%
Best model saved with acc 1.4925%

 Fold 1, Epoch 2/50


100%|██████████| 3/3 [00:00<00:00,  4.44batch/s]


Epoch 2: Train Loss: 4.4651, Train Acc: 3.6232% | Val Loss: 4.6673, Val Acc: 5.9701%, F1: 0.0402
Top-1 Accuracy: 5.9701% | Top-5 Accuracy: 10.4478% | Top-10 Accuracy: 16.4179% | Top-30 Accuracy: 32.8358%
Best model saved with acc 5.9701%

 Fold 1, Epoch 3/50


...

100%|██████████| 3/3 [00:00<00:00,  4.93batch/s]



 Fold 5, Epoch 50/50


100%|██████████| 3/3 [00:00<00:00,  4.53batch/s]


 Average across 5 folds:
 - Accuracy: 18.12%
Top-1 Accuracy: 17.2185% | Top-5 Accuracy: 30.5201% | Top-10 Accuracy: 40.4930% | Top-30 Accuracy: 61.0312%


[{'val_acc': 19.402985074626866,
  'val_topk': {1: 17.91044771671295,
   5: 28.358209133148193,
   10: 37.31343150138855,
   30: 59.70149040222168}},
 {'val_acc': 22.727272727272727,
  'val_topk': {1: 22.727273404598236,
   5: 34.84848439693451,
   10: 43.939393758773804,
   30: 59.090906381607056}},
 {'val_acc': 18.181818181818183,
  'val_topk': {1: 16.66666716337204,
   5: 31.81818127632141,
   10: 42.424243688583374,
   30: 63.63636255264282}},
 {'val_acc': 16.666666666666664,
  'val_topk': {1: 16.66666716337204,
   5: 31.81818127632141,
   10: 39.393940567970276,
   30: 63.63636255264282}},
 {'val_acc': 13.636363636363635,
  'val_topk': {1: 12.121212482452393,
   5: 25.757575035095215,
   10: 39.393940567970276,
   30: 59.090906381607056}}]

In [ ]:
# swin_T test
test_metrics = []
for fold in range(5):
    num_classes = len(disease_mappings)
    device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
    backbone = models.swin_t(weights="Swin_T_Weights.IMAGENET1K_V1")
    test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
    test_model.load_state_dict(torch.load(f'logs_new/SWIN_T/fold{fold+1}_best.pth'))
    test_model.to(device)
    test_model.eval()  # Set model to evaluation mode
    criterion = nn.CrossEntropyLoss()
    val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
    print(val_acc)
    print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

    test_metrics.append({
        "val_acc": val_acc,
        "val_topk": topk_accuracies
    })

# Compute average and std of overall accuracy
val_accs = [m["val_acc"] for m in test_metrics]
avg_acc = np.mean(val_accs)
std_acc = np.std(val_accs, ddof=1)

# Compute average and std for each top-k accuracy
avg_topk = {}
std_topk = {}
for k in test_metrics[0]["val_topk"].keys():
    scores = [m["val_topk"][k] for m in test_metrics]
    avg_topk[k] = np.mean(scores)
    std_topk[k] = np.std(scores, ddof=1)

# Print results
print(f"\nAverage across {len(test_metrics)} folds:")
print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

for k in sorted(avg_topk.keys(), key=int):
    print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")

100%|██████████| 4/4 [00:00<00:00,  4.94batch/s]


10.619469026548673
Top-1 Accuracy: 10.6195% | Top-5 Accuracy: 24.7788% | Top-10 Accuracy: 36.2832% | Top-30 Accuracy: 59.2920%


100%|██████████| 4/4 [00:00<00:00,  5.56batch/s]


17.699115044247787
Top-1 Accuracy: 17.6991% | Top-5 Accuracy: 24.7788% | Top-10 Accuracy: 35.3982% | Top-30 Accuracy: 57.5221%


100%|██████████| 4/4 [00:00<00:00,  5.56batch/s]


15.04424778761062
Top-1 Accuracy: 15.0442% | Top-5 Accuracy: 27.4336% | Top-10 Accuracy: 35.3982% | Top-30 Accuracy: 63.7168%


100%|██████████| 4/4 [00:00<00:00,  5.34batch/s]


15.04424778761062
Top-1 Accuracy: 15.0442% | Top-5 Accuracy: 29.2035% | Top-10 Accuracy: 40.7080% | Top-30 Accuracy: 58.4071%


100%|██████████| 4/4 [00:00<00:00,  5.48batch/s]

13.274336283185843
Top-1 Accuracy: 13.2743% | Top-5 Accuracy: 24.7788% | Top-10 Accuracy: 31.8584% | Top-30 Accuracy: 53.0973%

Average across 5 folds:
 - Accuracy: 14.34% ± 2.61% (1-sigma)
 - Top-1 Accuracy: 14.34% ± 2.61% (1-sigma)
 - Top-5 Accuracy: 26.19% ± 2.04% (1-sigma)
 - Top-10 Accuracy: 35.93% ± 3.17% (1-sigma)
 - Top-30 Accuracy: 58.41% ± 3.81% (1-sigma)


In [ ]:
# CLIP
num_classes = len(disease_mappings)
save_dir = "logs_new/CLIP"
device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
clip_model, _= clip.load("ViT-B/32", device=device, jit=False)
clip_model.visual = clip_model.visual.float()
backbone = clip_model.visual
model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)



 Fold 1

 Fold 1, Epoch 1/50


100%|██████████| 3/3 [00:00<00:00,  4.34batch/s]


Epoch 1: Train Loss: 4.9673, Train Acc: 1.4493% | Val Loss: 4.6285, Val Acc: 1.4925%, F1: 0.0005
Top-1 Accuracy: 1.4925% | Top-5 Accuracy: 4.4776% | Top-10 Accuracy: 7.4627% | Top-30 Accuracy: 31.3433%
Best model saved with acc 1.4925%

 Fold 1, Epoch 2/50


100%|██████████| 3/3 [00:00<00:00,  4.49batch/s]



 Fold 1, Epoch 3/50


...

100%|██████████| 3/3 [00:02<00:00,  1.10batch/s]



 Fold 5, Epoch 50/50


100%|██████████| 3/3 [00:02<00:00,  1.16batch/s]


 Average across 5 folds:
 - Accuracy: 6.95%
Top-1 Accuracy: 3.9258% | Top-5 Accuracy: 11.1895% | Top-10 Accuracy: 15.7214% | Top-30 Accuracy: 41.6780%


[{'val_acc': 5.970149253731343,
  'val_topk': {1: 4.477611929178238,
   5: 7.46268630027771,
   10: 11.940298229455948,
   30: 46.268656849861145}},
 {'val_acc': 7.575757575757576,
  'val_topk': {1: 1.515151560306549,
   5: 12.121212482452393,
   10: 16.66666716337204,
   30: 40.909090638160706}},
 {'val_acc': 7.575757575757576,
  'val_topk': {1: 6.060606241226196,
   5: 15.15151560306549,
   10: 19.696970283985138,
   30: 51.51515007019043}},
 {'val_acc': 6.0606060606060606,
  'val_topk': {1: 3.030303120613098,
   5: 12.121212482452393,
   10: 15.15151560306549,
   30: 30.30303120613098}},
 {'val_acc': 7.575757575757576,
  'val_topk': {1: 4.545454680919647,
   5: 9.090909361839294,
   10: 15.15151560306549,
   30: 39.393940567970276}}]

In [ ]:
# clip test
test_metrics = []
for fold in range(5):
    num_classes = len(disease_mappings)
    device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
    clip_model, _= clip.load("ViT-B/32", device=device, jit=False)
    clip_model.visual = clip_model.visual.float()
    backbone = clip_model.visual
    test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
    test_model.load_state_dict(torch.load(f'logs_new/CLIP/fold{fold+1}_best.pth'))
    test_model.to(device)
    test_model.eval()  # Set model to evaluation mode
    criterion = nn.CrossEntropyLoss()
    val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
    print(val_acc)
    print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

    test_metrics.append({
        "val_acc": val_acc,
        "val_topk": topk_accuracies
    })

# Compute average and std of overall accuracy
val_accs = [m["val_acc"] for m in test_metrics]
avg_acc = np.mean(val_accs)
std_acc = np.std(val_accs, ddof=1)

# Compute average and std for each top-k accuracy
avg_topk = {}
std_topk = {}
for k in test_metrics[0]["val_topk"].keys():
    scores = [m["val_topk"][k] for m in test_metrics]
    avg_topk[k] = np.mean(scores)
    std_topk[k] = np.std(scores, ddof=1)

# Print results
print(f"\nAverage across {len(test_metrics)} folds:")
print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

for k in sorted(avg_topk.keys(), key=int):
    print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")

100%|██████████| 4/4 [00:00<00:00,  5.86batch/s]


2.6548672566371683
Top-1 Accuracy: 2.6549% | Top-5 Accuracy: 14.1593% | Top-10 Accuracy: 20.3540% | Top-30 Accuracy: 45.1327%


100%|██████████| 4/4 [00:00<00:00,  4.88batch/s]


1.7699115044247788
Top-1 Accuracy: 1.7699% | Top-5 Accuracy: 10.6195% | Top-10 Accuracy: 14.1593% | Top-30 Accuracy: 38.0531%


100%|██████████| 4/4 [00:00<00:00,  5.73batch/s]


1.7699115044247788
Top-1 Accuracy: 1.7699% | Top-5 Accuracy: 9.7345% | Top-10 Accuracy: 13.2743% | Top-30 Accuracy: 37.1681%


100%|██████████| 4/4 [00:00<00:00,  5.66batch/s]


3.5398230088495577
Top-1 Accuracy: 3.5398% | Top-5 Accuracy: 16.8142% | Top-10 Accuracy: 24.7788% | Top-30 Accuracy: 44.2478%


100%|██████████| 4/4 [00:00<00:00,  6.29batch/s]

5.3097345132743365
Top-1 Accuracy: 5.3097% | Top-5 Accuracy: 12.3894% | Top-10 Accuracy: 23.0089% | Top-30 Accuracy: 46.9027%

Average across 5 folds:
 - Accuracy: 3.01% ± 1.48% (1-sigma)
 - Top-1 Accuracy: 3.01% ± 1.48% (1-sigma)
 - Top-5 Accuracy: 12.74% ± 2.84% (1-sigma)
 - Top-10 Accuracy: 19.12% ± 5.18% (1-sigma)
 - Top-30 Accuracy: 42.30% ± 4.40% (1-sigma)


In [ ]:
# VGGFACE
num_classes = len(disease_mappings)
save_dir = "logs_new/VGGFACE"
device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
vgg = vgg_face('models_vggface/vgg_face.pth')
backbone = VggFaceFeatureExtractor(vgg)
model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False, conv_out_channels=512)
train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)



 Fold 1

 Fold 1, Epoch 1/50


100%|██████████| 3/3 [00:00<00:00,  3.79batch/s]


Epoch 1: Train Loss: 4.7609, Train Acc: 3.2609% | Val Loss: 4.6610, Val Acc: 0.0000%, F1: 0.0000
Top-1 Accuracy: 0.0000% | Top-5 Accuracy: 5.9701% | Top-10 Accuracy: 8.9552% | Top-30 Accuracy: 25.3731%
Best model saved with acc 0.0000%

 Fold 1, Epoch 2/50


100%|██████████| 3/3 [00:00<00:00,  3.04batch/s]


Epoch 2: Train Loss: 3.8606, Train Acc: 19.5652% | Val Loss: 4.5147, Val Acc: 0.0000%, F1: 0.0000
Top-1 Accuracy: 0.0000% | Top-5 Accuracy: 11.9403% | Top-10 Accuracy: 14.9254% | Top-30 Accuracy: 38.8060%
Best model saved with acc 0.0000%

 Fold 1, Epoch 3/50


...

100%|██████████| 3/3 [00:00<00:00,  3.88batch/s]



 Fold 5, Epoch 50/50


100%|██████████| 3/3 [00:00<00:00,  3.77batch/s]


 Average across 5 folds:
 - Accuracy: 11.79%
Top-1 Accuracy: 9.9729% | Top-5 Accuracy: 28.7155% | Top-10 Accuracy: 38.0642% | Top-30 Accuracy: 57.1099%


[{'val_acc': 8.955223880597014,
  'val_topk': {1: 8.955223858356476,
   5: 23.880596458911896,
   10: 38.805970549583435,
   30: 53.731346130371094}},
 {'val_acc': 12.121212121212121,
  'val_topk': {1: 10.606060922145844,
   5: 30.30303120613098,
   10: 40.909090638160706,
   30: 53.03030014038086}},
 {'val_acc': 16.666666666666664,
  'val_topk': {1: 13.636364042758942,
   5: 36.36363744735718,
   10: 42.424243688583374,
   30: 63.63636255264282}},
 {'val_acc': 9.090909090909092,
  'val_topk': {1: 7.575757801532745,
   5: 25.757575035095215,
   10: 34.84848439693451,
   30: 59.090906381607056}},
 {'val_acc': 12.121212121212121,
  'val_topk': {1: 9.090909361839294,
   5: 27.272728085517883,
   10: 33.33333432674408,
   30: 56.060606241226196}}]

In [ ]:
# vggface test
test_metrics = []
for fold in range(5):
    num_classes = len(disease_mappings)
    device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
    vgg = vgg_face('models_vggface/vgg_face.pth')
    backbone = VggFaceFeatureExtractor(vgg)
    test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False, conv_out_channels=512)
    test_model.load_state_dict(torch.load(f'logs_new/VGGFACE/fold{fold+1}_best.pth'))
    test_model.to(device)
    test_model.eval()  # Set model to evaluation mode
    criterion = nn.CrossEntropyLoss()
    val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
    print(val_acc)
    print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

    test_metrics.append({
        "val_acc": val_acc,
        "val_topk": topk_accuracies
    })

# Compute average and std of overall accuracy
val_accs = [m["val_acc"] for m in test_metrics]
avg_acc = np.mean(val_accs)
std_acc = np.std(val_accs, ddof=1)

# Compute average and std for each top-k accuracy
avg_topk = {}
std_topk = {}
for k in test_metrics[0]["val_topk"].keys():
    scores = [m["val_topk"][k] for m in test_metrics]
    avg_topk[k] = np.mean(scores)
    std_topk[k] = np.std(scores, ddof=1)

# Print results
print(f"\nAverage across {len(test_metrics)} folds:")
print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

for k in sorted(avg_topk.keys(), key=int):
    print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")

100%|██████████| 4/4 [00:00<00:00,  5.45batch/s]


11.504424778761061
Top-1 Accuracy: 11.5044% | Top-5 Accuracy: 28.3186% | Top-10 Accuracy: 38.9381% | Top-30 Accuracy: 59.2920%


100%|██████████| 4/4 [00:00<00:00,  5.20batch/s]


11.504424778761061
Top-1 Accuracy: 11.5044% | Top-5 Accuracy: 26.5487% | Top-10 Accuracy: 38.0531% | Top-30 Accuracy: 62.8319%


100%|██████████| 4/4 [00:00<00:00,  5.85batch/s]


14.15929203539823
Top-1 Accuracy: 14.1593% | Top-5 Accuracy: 30.9735% | Top-10 Accuracy: 38.9381% | Top-30 Accuracy: 61.0619%


100%|██████████| 4/4 [00:00<00:00,  5.25batch/s]


11.504424778761061
Top-1 Accuracy: 11.5044% | Top-5 Accuracy: 33.6283% | Top-10 Accuracy: 39.8230% | Top-30 Accuracy: 62.8319%


100%|██████████| 4/4 [00:00<00:00,  4.83batch/s]

9.734513274336283
Top-1 Accuracy: 9.7345% | Top-5 Accuracy: 30.0885% | Top-10 Accuracy: 36.2832% | Top-30 Accuracy: 58.4071%

Average across 5 folds:
 - Accuracy: 11.68% ± 1.58% (1-sigma)
 - Top-1 Accuracy: 11.68% ± 1.58% (1-sigma)
 - Top-5 Accuracy: 29.91% ± 2.68% (1-sigma)
 - Top-10 Accuracy: 38.41% ± 1.34% (1-sigma)
 - Top-30 Accuracy: 60.88% ± 2.02% (1-sigma)
